# The Price Is Right — Simplified
Predict Amazon product prices from descriptions using GPT-4o-mini.

Steps:
1. Load a small sample of Amazon product data
2. Baseline: always guess the average price
3. LLM: ask GPT-4o-mini to guess the price
4. Compare both with Mean Absolute Error
5. Gradio UI to test with your own product description

In [ ]:
import re
import random
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
from datasets import load_dataset
from tqdm import tqdm

load_dotenv(override=True)
client = OpenAI()
MODEL = 'gpt-4o-mini'

In [ ]:
# Load a small sample of Amazon Appliances data

dataset = load_dataset(
    'McAuley-Lab/Amazon-Reviews-2023',
    'raw_meta_Appliances',
    split='full',
    trust_remote_code=True
)

print(f'Total items: {len(dataset):,}')

In [ ]:
# Clean and filter items with valid prices between $1-$1000 and a description

def clean(item):
    try:
        price = float(item['price'])
    except (ValueError, TypeError):
        return None
    if not (1 <= price <= 1000):
        return None
    title = item.get('title', '').strip()
    features = ' '.join(item.get('features', []))
    description = ' '.join(item.get('description', []))
    text = f"{title}. {features} {description}".strip()
    if len(text) < 30:
        return None
    return {'text': text[:1000], 'price': price}

items = [clean(d) for d in dataset]
items = [i for i in items if i]
print(f'Clean items: {len(items):,}')

In [ ]:
# Take a small random sample for evaluation (to keep API costs low)

random.seed(42)
sample = random.sample(items, min(50, len(items)))

prices = [i['price'] for i in sample]
avg_price = sum(prices) / len(prices)
print(f'Sample size: {len(sample)}')
print(f'Average price: ${avg_price:.2f}')
print(f'Min: ${min(prices):.2f}  Max: ${max(prices):.2f}')

In [ ]:
# --- Baseline: always guess the average price ---

def mae(predictions, actuals):
    return sum(abs(p - a) for p, a in zip(predictions, actuals)) / len(actuals)

baseline_preds = [avg_price] * len(sample)
actuals = [i['price'] for i in sample]
baseline_mae = mae(baseline_preds, actuals)
print(f'Baseline MAE (always guess avg): ${baseline_mae:.2f}')

In [ ]:
# --- LLM price prediction ---

SYSTEM = """You are a pricing expert. Given a product description, predict its price in USD.
Reply with ONLY a number. No dollar sign, no explanation. Just the number."""

def predict_price(text):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Product: {text}\n\nPrice:"},
        ],
        max_tokens=10,
    )
    raw = response.choices[0].message.content.strip()
    match = re.search(r'[\d.]+', raw)
    return float(match.group()) if match else avg_price

print('Running LLM predictions...')
llm_preds = [predict_price(i['text']) for i in tqdm(sample)]
llm_mae = mae(llm_preds, actuals)
print(f'LLM MAE: ${llm_mae:.2f}')

In [ ]:
# --- Results comparison ---

print('=' * 40)
print(f'  Baseline MAE : ${baseline_mae:.2f}')
print(f'  LLM MAE     : ${llm_mae:.2f}')
improvement = ((baseline_mae - llm_mae) / baseline_mae) * 100
print(f'  Improvement  : {improvement:.1f}%')
print('=' * 40)

print('\nSample predictions:')
print(f'{"Actual":>10}  {"LLM Guess":>10}  {"Error":>10}')
for actual, pred in zip(actuals[:10], llm_preds[:10]):
    print(f'${actual:>9.2f}  ${pred:>9.2f}  ${abs(actual-pred):>9.2f}')

In [ ]:
# --- Gradio UI: test with your own description ---

def gradio_predict(description):
    if not description.strip():
        return 'Please enter a product description.'
    price = predict_price(description)
    return f'Predicted price: ${price:.2f}'

gr.Interface(
    fn=gradio_predict,
    inputs=gr.Textbox(lines=5, label='Product Description', placeholder='Paste an Amazon product description here...'),
    outputs=gr.Textbox(label='Predicted Price'),
    title='The Price Is Right',
    description='Enter a product description and GPT-4o-mini will predict its price.'
).launch()